In [ ]:
# CSBP441 Applied Computer Vision
# Week 1 - Assignment 1 Colab Code
#
# Purpose:
#   Image basics, OpenCV setup, grayscale conversion, brightness changes,
#   thresholding, output saving, and first result analysis.
#
# How to use in Google Colab:
#   1. Upload this .py file to Colab or copy the cells into a notebook.
#   2. Run from top to bottom.
#   3. Optionally upload two images when prompted.
#   4. Check the generated outputs/ folder.



# Assignment 1 - Image Basics and OpenCV

This code supports the assignment cycle:

**hand calculation -> OpenCV/Python implementation -> result analysis -> real-system interpretation**



## Setup in Google Colab

OpenCV, NumPy, and Matplotlib are usually already installed in Colab.
Run the next cell only if `import cv2` fails.



In [ ]:
# Uncomment this line in Colab only if OpenCV is missing:
# !pip install opencv-python



In [ ]:
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np


OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("OpenCV version:", cv2.__version__)
print("Output folder:", OUTPUT_DIR.resolve())




## Part A - Hand Calculation Check

This is the 5x5 grayscale image from the assignment. The code below computes
values students should also compute by hand.



In [ ]:
I = np.array(
    [
        [10, 10, 30, 60, 60],
        [10, 20, 35, 65, 70],
        [15, 25, 40, 80, 85],
        [90, 90, 95, 100, 105],
        [95, 100, 100, 110, 120],
    ],
    dtype=np.uint8,
)

T = 50
binary_I = (I >= T).astype(np.uint8)

print("Matrix I:")
print(I)
print("\nMinimum intensity:", I.min())
print("Maximum intensity:", I.max())
print("Mean intensity:", I.mean())
print("Row averages:", I.mean(axis=1))
print(f"\nBinary image with threshold T = {T}:")
print(binary_I)

plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1)
plt.imshow(I, cmap="gray", vmin=0, vmax=255)
plt.title("Original 5x5 Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(binary_I, cmap="gray", vmin=0, vmax=1)
plt.title(f"Threshold T={T}")
plt.axis("off")
plt.tight_layout()
plt.show()




## Part B - Load Two Images

In Colab, you can upload two images. If you skip upload or run outside Colab,
the code creates two synthetic test images so the assignment can still run.



In [ ]:
def create_synthetic_images():
    """Create two simple test images when no upload is available."""
    h, w = 320, 420

    # Synthetic "natural scene": sky/ground gradient with sun and hill.
    natural = np.zeros((h, w, 3), dtype=np.uint8)
    for y in range(h):
        if y < h // 2:
            natural[y, :, :] = (180 - y // 4, 160 - y // 5, 80 + y // 3)  # BGR sky
        else:
            natural[y, :, :] = (70, 120 + (y - h // 2) // 3, 60)
    cv2.circle(natural, (330, 70), 35, (30, 220, 255), -1)
    cv2.ellipse(natural, (170, 230), (170, 60), 0, 0, 360, (40, 90, 40), -1)

    # Synthetic "object/document": bright paper with dark text-like lines.
    document = np.full((h, w, 3), 235, dtype=np.uint8)
    cv2.rectangle(document, (55, 35), (365, 285), (250, 250, 250), -1)
    cv2.rectangle(document, (55, 35), (365, 285), (40, 40, 40), 3)
    for i, y in enumerate(range(75, 245, 35)):
        cv2.line(document, (85, y), (330 - 20 * (i % 3), y), (35, 35, 35), 3)
    cv2.circle(document, (105, 250), 18, (50, 50, 180), -1)

    cv2.imwrite(str(OUTPUT_DIR / "synthetic_natural.png"), natural)
    cv2.imwrite(str(OUTPUT_DIR / "synthetic_document.png"), document)
    return {
        "natural_scene": natural,
        "object_document": document,
    }


def load_uploaded_images_or_fallback():
    uploaded_images = {}

    try:
        from google.colab import files  # type: ignore

        print("Upload two images for the assignment.")
        print("If you cancel upload, synthetic images will be used.")
        uploaded = files.upload()

        for file_name in uploaded:
            data = np.frombuffer(uploaded[file_name], np.uint8)
            img = cv2.imdecode(data, cv2.IMREAD_COLOR)
            if img is not None:
                key = Path(file_name).stem
                uploaded_images[key] = img
    except Exception as exc:
        print("Upload not available or skipped:", exc)

    if len(uploaded_images) >= 2:
        print(f"Loaded {len(uploaded_images)} uploaded image(s).")
        return dict(list(uploaded_images.items())[:2])

    print("Using synthetic test images.")
    return create_synthetic_images()


images = load_uploaded_images_or_fallback()
print("Image keys:", list(images.keys()))




## Helper Functions



In [ ]:
def show_bgr(img_bgr, title="", cmap=None):
    """Display BGR OpenCV image correctly in matplotlib."""
    plt.figure(figsize=(5, 4))
    if img_bgr.ndim == 2:
        plt.imshow(img_bgr, cmap=cmap or "gray", vmin=0, vmax=255)
    else:
        plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()


def save_image(name, img):
    path = OUTPUT_DIR / name
    cv2.imwrite(str(path), img)
    print("Saved:", path)


def describe_image(name, img):
    print(f"\n{name}")
    print("- shape:", img.shape)
    print("- dtype:", img.dtype)
    print("- min/max:", img.min(), img.max())
    if img.ndim == 3:
        h, w, c = img.shape
        print(f"- height={h}, width={w}, channels={c}")
        print("- OpenCV channel order: BGR")
    else:
        h, w = img.shape
        print(f"- height={h}, width={w}, grayscale image")




## Part B - OpenCV/Python Implementation



In [ ]:
threshold_values = [80, 130]

results = {}

for name, img in images.items():
    describe_image(name, img)
    show_bgr(img, f"{name} - original")
    save_image(f"{name}_original.png", img)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    describe_image(f"{name}_gray", gray)
    show_bgr(gray, f"{name} - grayscale")
    save_image(f"{name}_gray.png", gray)

    brighter = cv2.add(gray, 50)
    darker = cv2.subtract(gray, 50)
    save_image(f"{name}_brighter_plus50.png", brighter)
    save_image(f"{name}_darker_minus50.png", darker)

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.imshow(darker, cmap="gray", vmin=0, vmax=255)
    plt.title("Darker (-50)")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(gray, cmap="gray", vmin=0, vmax=255)
    plt.title("Grayscale")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(brighter, cmap="gray", vmin=0, vmax=255)
    plt.title("Brighter (+50)")
    plt.axis("off")
    plt.suptitle(name)
    plt.tight_layout()
    plt.show()

    thresholds_for_image = {}
    for T in threshold_values:
        _, binary = cv2.threshold(gray, T, 255, cv2.THRESH_BINARY)
        save_image(f"{name}_threshold_T{T}.png", binary)
        thresholds_for_image[T] = binary

    results[name] = {
        "original": img,
        "gray": gray,
        "brighter": brighter,
        "darker": darker,
        "thresholds": thresholds_for_image,
    }




## Compare Threshold Values



In [ ]:
for name, data in results.items():
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.imshow(data["gray"], cmap="gray", vmin=0, vmax=255)
    plt.title("Grayscale")
    plt.axis("off")

    for idx, T in enumerate(threshold_values, start=2):
        plt.subplot(1, 3, idx)
        plt.imshow(data["thresholds"][T], cmap="gray", vmin=0, vmax=255)
        plt.title(f"Threshold T={T}")
        plt.axis("off")

    plt.suptitle(f"Threshold comparison: {name}")
    plt.tight_layout()
    plt.show()




## Part C - Result Analysis Prompts

Students should answer these in the PDF report.



In [ ]:
analysis_prompts = [
    "1. Compare the original and grayscale images. What information is lost?",
    "2. Compare two threshold values. Which threshold works better for each image and why?",
    "3. Describe one case where thresholding fails.",
    "4. Explain one parameter or image condition that strongly affected the result.",
]

print("\nResult Analysis Prompts")
print("-" * 30)
for prompt in analysis_prompts:
    print(prompt)




## Part D - Real-System Interpretation Template



In [ ]:
pipeline_template = {
    "Application": "Example: document scanning / coin counting / traffic monitoring / drone crop monitoring",
    "Input": "Camera image or video",
    "Preprocessing": "Resize, grayscale conversion, blur, brightness/contrast correction",
    "Main algorithm/model": "Thresholding, edges/contours, or a deep learning model",
    "Postprocessing": "Morphology, connected components, contour filtering, NMS, or tracking",
    "Decision/output": "Count, label, detection boxes, segmentation mask, or alert",
}

print("\nReal-System Interpretation Template")
print("-" * 40)
for stage, description in pipeline_template.items():
    print(f"{stage}: {description}")




## Optional: Zip Outputs for Download



In [ ]:
try:
    import shutil

    zip_path = shutil.make_archive("assignment1_outputs", "zip", OUTPUT_DIR)
    print("Created zip file:", zip_path)

    try:
        from google.colab import files  # type: ignore

        files.download(zip_path)
    except Exception:
        print("Download helper is only available in Colab.")
except Exception as exc:
    print("Could not create zip file:", exc)
